In [1]:
import os, re, time, warnings, logging, json
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
log = logging.getLogger(__name__)

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm.auto import tqdm
tqdm.pandas()

for r in ['stopwords', 'wordnet', 'punkt', 'omw-1.4']:
    nltk.download(r, quiet=True)

ROOT      = Path('/kaggle/working')          
DATA_IN   = Path('/kaggle/input/notebooks/bennjimatakwa/datamerge01/data/processed/merged_reviews.csv')
DATA_OUT  = ROOT / 'data' / 'processed'
MODEL_OUT = ROOT / 'models'
FIG_OUT   = ROOT / 'figures'

for d in [DATA_OUT, MODEL_OUT, FIG_OUT]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    'random_state'    : 42,
    'sample_size'     : 100_000,
    'min_word_count'  : 2,
    'recency_halflife': 365,
    'max_review_len'  : 500,
    'sentiment_map'   : {1: 'negative', 2: 'negative', 3: 'neutral',
                         4: 'positive', 5: 'positive'},
    'label_map'       : {'negative': 0, 'neutral': 1, 'positive': 2}
}
np.random.seed(CFG['random_state'])

print('✅ Imports done')
print(f'ROOT     : {ROOT}')
print(f'DATA_OUT : {DATA_OUT}')
print(f'MODEL_OUT: {MODEL_OUT}')

2026-05-12 16:58:55,424 | INFO | NumExpr defaulting to 4 threads.


✅ Imports done
ROOT     : /kaggle/working
DATA_OUT : /kaggle/working/data/processed
MODEL_OUT: /kaggle/working/models


# Load Processed Dataset

In [2]:
log.info('Loading dataset...')
t0 = time.time()

df = pd.read_csv(DATA_IN, low_memory=False)

log.info(f'Loaded in {time.time()-t0:.1f}s')

assert 'content' in df.columns
assert 'score'   in df.columns
assert 'appName' in df.columns
assert df['content'].isna().sum() <= 10
assert df['score'].between(1, 5).all()

print(f'Shape     : {df.shape}')
print(f'Platforms : {sorted(df["appName"].unique())}')
print(f'Score range: {df["score"].min()} → {df["score"].max()}')
df.head(3)

2026-05-12 16:58:59,670 | INFO | Loading dataset...
2026-05-12 16:59:03,581 | INFO | Loaded in 3.9s


Shape     : (630000, 8)
Platforms : ['Alibaba', 'Aliexpress', 'Amazon shopping', 'Daraz Online Shopping App', 'Flipkart', 'Lazada', 'Meesho', 'Myntra', 'Shein', 'Snapdeal', 'Walmart']
Score range: 1 → 5


,reviewId,content,score,thumbsUpCount,at,replyContent,repliedAt,appName
0,275f465b-a58b-439e-ae7c-f9f6dcf2634d,Trying to use the on website is almost impossi...,1,39,1720995717000,"Hi, we are sorry to hear that. Do share additi...",1.721048e+12,Alibaba
1,e6c13852-277e-451a-b8d5-dd92aea75402,Had to uninstall due to the amount of notifica...,3,60,1720501958000,"Hi, we are sorry to hear that. Do share additi...",1.721051e+12,Alibaba
2,254b3705-c54b-4ce4-8982-5b468d38231d,I order and it takes too long the shpping days...,1,7,1721866371000,NaN,NaN,Alibaba


In [3]:
STOPWORDS = set(stopwords.words('english'))
KEEP_WORDS = {'no','not','nor','never','nothing','neither',
              'nobody','nowhere','cannot',"can't","won't",
              "don't","doesn't","didn't","isn't","wasn't",
              "aren't","weren't","hasn't","haven't","hadn't"}
STOPWORDS -= KEEP_WORDS
lemmatizer = WordNetLemmatizer()

def clean_text(text) -> str:
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    emoji_pat = re.compile('['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+', flags=re.UNICODE)
    text = emoji_pat.sub(' ', text)
    text = re.sub(r'[^a-z\s\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [lemmatizer.lemmatize(t) for t in text.split()
              if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

# Quick test
tests = ["I LOVE this app!! 😍 https://example.com",
         "Worst app. Can't get refund. Don't buy!", "  ", None]
for t in tests:
    print(f'{str(t)[:45]:<45} → {clean_text(t)}')

I LOVE this app!! 😍 https://example.com       → love app
Worst app. Can't get refund. Don't buy!       → worst app can't get refund don't buy
                                              → 
None                                          → 


##  Text Cleaning Pipeline

| Step | What it removes | Example |
|------|----------------|---------|
| Lowercase | Inconsistent casing | `"LOVE"` → `"love"` |
| URL removal | Links (no semantic value) | `https://...` → ` ` |
| Email removal | Personal data | `user@mail.com` → ` ` |
| Emoji removal | Unicode symbols | `😍` → ` ` |
| Special chars | Punctuation, numbers | `"3/5!!"` → `" "` |
| Stopword removal | Common words | `"the", "is", "and"` → remo

# APPLY cleaning 

In [4]:
log.info('Cleaning text...')
t0 = time.time()

df['clean_content'] = df['content'].progress_apply(clean_text)

before = len(df)
df = df[df['clean_content'].str.split().str.len() >= CFG['min_word_count']]
df = df.reset_index(drop=True)

log.info(f'Done in {time.time()-t0:.1f}s')
print(f'Rows removed : {before - len(df):,}')
print(f'Remaining    : {len(df):,}')

2026-05-12 16:59:06,195 | INFO | Cleaning text...


  0%|          | 0/630000 [00:00<?, ?it/s]

2026-05-12 17:00:06,218 | INFO | Done in 60.0s


Rows removed : 5,986
Remaining    : 624,014


# sentiment labels 
This cell creates labels for supervised learning based on review scores.


In [5]:
df['sentiment']    = df['score'].map(CFG['sentiment_map'])
df['sentiment_id'] = df['sentiment'].map(CFG['label_map'])

assert df['sentiment'].isna().sum() == 0
assert set(df['sentiment'].unique()) == {'negative', 'neutral', 'positive'}

counts = df['sentiment'].value_counts()
total  = len(df)

print('Sentiment distribution:')
for label, count in counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<10} {count:>8,}  ({pct:5.1f}%)  {bar}')

pos = counts.get('positive', 0)
neu = counts.get('neutral',  0)
neg = counts.get('negative', 0)
print(f'\nPositive/Neutral = {pos/neu:.1f}x | Negative/Neutral = {neg/neu:.1f}x')

Sentiment distribution:
  positive    330,194  ( 52.9%)  ██████████████████████████
  negative    253,975  ( 40.7%)  ████████████████████
  neutral      39,845  (  6.4%)  ███

Positive/Neutral = 8.3x | Negative/Neutral = 6.4x



**Label mapping:**
| Stars | Sentiment | ID | Count | % |
|-------|-----------|----|-------|---|
| 1–2★ | negative | 0 | 253,975 | 40.4% |
| 3★ | neutral | 1 | 39,845 | 6.4% |
| 4–5★ | positive | 2 | 330,194 | 53.2% |


# feature engineering 

In [6]:
log.info('Engineering text features...')

df['review_length']     = df['content'].str.len().clip(0, CFG['max_review_len'])
df['word_count']        = df['clean_content'].str.split().str.len().fillna(0).astype(int)
df['char_per_word']     = (df['review_length'] / (df['word_count'] + 1)).round(2)
df['unique_word_ratio'] = df['clean_content'].apply(
    lambda x: len(set(x.split())) / (len(x.split()) + 1)
    if isinstance(x, str) else 0).round(4)
df['exclamation_count'] = df['content'].str.count('!').fillna(0).astype(int)
df['question_count']    = df['content'].str.count('\?').fillna(0).astype(int)
df['caps_ratio']        = df['content'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1)).round(4)

print('Text features done')
print(df[['review_length','word_count','exclamation_count']].describe().round(2))

2026-05-12 17:00:06,631 | INFO | Engineering text features...


Text features done
       review_length  word_count  exclamation_count
count      624014.00   624014.00          624014.00
mean          164.90       16.40               0.29
std           136.45       13.11               1.14
min             8.00        2.00               0.00
25%            54.00        6.00               0.00
50%           122.00       12.00               0.00
75%           239.00       23.00               0.00
max           500.00      190.00              68.00


## Text Feature Engineering

Extracts 7 quantitative features from the raw and cleaned review text that capture writing style, expressiveness, and engagement signals beyond just the words themselves.

**Features engineered:**

| Feature | Formula | What it captures |
|---------|---------|-----------------|
| `review_length` | `len(content)` clipped at 500 | Overall verbosity |
| `word_count` | `len(clean_content.split())` | Review depth |
| `char_per_word` | `review_length / (word_count + 1)` | Vocabulary complexity |
| `unique_word_ratio` | `unique_words / total_words` | Vocabulary diversity |
| `exclamation_count` | `content.count('!')` | Emotional intensity |
| `question_count` | `content.count('?')` | Confusion / inquiry |
| `caps_ratio` | `uppercase_chars / total_chars` | Shouting / anger |

**Key statistics:**
- Average review length: **164.9 characters** (~16 words)
- 75% of reviews have **0 exclamation marks** (calm tone)
- Maximum exclamation count: **68** (extreme frustration)
- Reviews range from 2 to **190 words** after cleaning

**Research insight:** Negative reviews tend to be longer (`complaint_depth` feature built later) and contain more exclamation marks — these stylometric features help models distinguish frustrated from satisfied customers even before reading the words.

In [7]:
log.info('Engineering temporal features...')

df['at'] = pd.to_datetime(df['at'], unit='ms', errors='coerce')

df['review_year']      = df['at'].dt.year
df['review_month']     = df['at'].dt.month
df['review_day']       = df['at'].dt.day
df['review_dayofweek'] = df['at'].dt.dayofweek
df['review_hour']      = df['at'].dt.hour
df['review_quarter']   = df['at'].dt.quarter
df['is_weekend']       = (df['review_dayofweek'] >= 5).astype(int)

# Cyclical encoding
df['hour_sin']  = np.sin(2 * np.pi * df['review_hour']  / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['review_hour']  / 24)
df['month_sin'] = np.sin(2 * np.pi * df['review_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['review_month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['review_dayofweek'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['review_dayofweek'] / 7)

print(f'Temporal features done | Year: {df["review_year"].min()} → {df["review_year"].max()}')

2026-05-12 17:00:15,376 | INFO | Engineering temporal features...


Temporal features done | Year: 2018 → 2024


In [8]:
log.info('Engineering weight features...')

max_date = df['at'].max()
df['days_ago']       = (max_date - df['at']).dt.days.fillna(0)
df['recency_weight'] = np.exp(-df['days_ago'] / CFG['recency_halflife']).round(6)

df['thumbsUpCount']  = df['thumbsUpCount'].fillna(0).clip(lower=0)
df['influence_weight'] = np.log1p(df['thumbsUpCount']).round(4)

p99 = df['influence_weight'].quantile(0.99)
df['influence_weight'] = df['influence_weight'].clip(upper=p99)

df['importance']     = (df['recency_weight'] * (1 + df['influence_weight'])).round(6)
df['weighted_score'] = (df['score'] * df['importance']).round(4)

print(f' Weight features done')
print(f'   recency_weight: {df["recency_weight"].min():.4f} → {df["recency_weight"].max():.4f}')
print(f'   influence p99 : {p99:.4f}')

2026-05-12 17:00:15,720 | INFO | Engineering weight features...


 Weight features done
   recency_weight: 0.0026 → 1.0000
   influence p99 : 5.2257


In [9]:
log.info('Engineering engagement + interaction features...')

df['at_clean']       = pd.to_datetime(df['at'],       errors='coerce')
df['repliedAt_clean']= pd.to_datetime(df['repliedAt'], unit='ms', errors='coerce')

df['has_reply']          = df['replyContent'].notna().astype(int)
df['response_time_hrs']  = ((df['repliedAt_clean'] - df['at_clean'])
                              .dt.total_seconds() / 3600).clip(0, 168)
df['is_auto_reply']      = ((df['has_reply'] == 1) &
                              (df['response_time_hrs'] < 0.5)).astype(int)
df['log_response_time']  = np.log1p(df['response_time_hrs'].fillna(0)).round(4)

# Interaction features
df['complaint_depth']   = (df['word_count'] * (df['score'] == 1).astype(int)).astype(int)
df['positive_brevity']  = ((1 / (1 + df['word_count'])) * (df['score'] == 5).astype(int)).round(4)
df['frustration_score'] = ((df['score'] == 1).astype(int) *
                            (1 + df['exclamation_count']) *
                            np.log1p(df['word_count'])).round(4)
df['engagement_quality']= (df['influence_weight'] * (df['score'] / 5)).round(4)

print(' Engagement + interaction features done')
print(f'   has_reply      : {df["has_reply"].sum():,} reviews have a reply')
print(f'   complaint_depth: mean={df["complaint_depth"].mean():.2f}')

2026-05-12 17:00:15,801 | INFO | Engineering engagement + interaction features...


 Engagement + interaction features done
   has_reply      : 165,926 reviews have a reply
   complaint_depth: mean=8.22


In [10]:
le_platform  = LabelEncoder()
le_sentiment = LabelEncoder()
df['platform_id']  = le_platform.fit_transform(df['appName'])
df['sentiment_id'] = le_sentiment.fit_transform(df['sentiment'])

NUMERICAL_FEATURES = [
    'word_count', 'char_per_word', 'unique_word_ratio',
    'exclamation_count', 'question_count', 'caps_ratio',
    'review_year', 'review_month', 'review_dayofweek', 'review_hour',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'recency_weight', 'influence_weight', 'importance', 'log_response_time',
    'complaint_depth', 'positive_brevity', 'frustration_score', 'engagement_quality'
]
BINARY_FEATURES    = ['has_reply', 'is_auto_reply', 'is_weekend']
ALL_MODEL_FEATURES = NUMERICAL_FEATURES + BINARY_FEATURES

feat_df         = df[NUMERICAL_FEATURES].fillna(0)
scaler_standard = StandardScaler()
scaler_minmax   = MinMaxScaler()
scaler_standard.fit_transform(feat_df)
scaler_minmax.fit_transform(feat_df)

print(f' Encoding + scaling done')
print(f'   Platforms : {list(le_platform.classes_)}')
print(f'   Sentiments: {list(le_sentiment.classes_)}')
print(f'   Features  : {len(ALL_MODEL_FEATURES)}')

 Encoding + scaling done
   Platforms : ['Alibaba', 'Aliexpress', 'Amazon shopping', 'Daraz Online Shopping App', 'Flipkart', 'Lazada', 'Meesho', 'Myntra', 'Shein', 'Snapdeal', 'Walmart']
   Sentiments: ['negative', 'neutral', 'positive']
   Features  : 27


In [11]:
# Compute class weights + 100k balanced sample 
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df['sentiment_id'].values
)
print('Class weights:')
for cls, w in zip(['negative','neutral','positive'], class_weights):
    print(f'  {cls:<10}: {w:.4f}')

# Balanced 100k sample
n_per_class    = min(CFG['sample_size'] // 3, df['sentiment'].value_counts().min())
sample = (df.groupby('sentiment', group_keys=False)
            .apply(lambda x: x.sample(n=n_per_class, random_state=CFG['random_state']))
            .sample(frac=1, random_state=CFG['random_state'])
            .reset_index(drop=True))

# Temporal train/val/test split
max_year  = sample['review_year'].max()
train_val = sample[sample['review_year'] < max_year].copy()
test      = sample[sample['review_year'] == max_year].copy()
if len(test) < len(sample) * 0.10:
    train_val, test = train_test_split(sample, test_size=0.15,
                        stratify=sample['sentiment_id'], random_state=CFG['random_state'])
train, val = train_test_split(train_val, test_size=0.12,
                stratify=train_val['sentiment_id'], random_state=CFG['random_state'])

print(f'\nSample : {len(sample):,} | Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

Class weights:
  negative  : 0.8190
  neutral   : 5.2203
  positive  : 0.6299

Sample : 99,999 | Train: 71,501 | Val: 9,751 | Test: 18,747


In [12]:
SAVE_COLUMNS = [c for c in [
    'reviewId', 'content', 'clean_content', 'score', 'appName',
    'at', 'thumbsUpCount', 'has_reply', 'replyContent',
    'sentiment', 'sentiment_id', 'platform_id',
    'review_length', 'word_count', 'char_per_word',
    'unique_word_ratio', 'exclamation_count', 'question_count', 'caps_ratio',
    'review_year', 'review_month', 'review_dayofweek', 'review_hour',
    'review_quarter', 'is_weekend',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'days_ago', 'recency_weight', 'influence_weight', 'importance', 'weighted_score',
    'response_time_hrs', 'log_response_time', 'is_auto_reply',
    'complaint_depth', 'positive_brevity', 'frustration_score', 'engagement_quality'
] if c in df.columns]

files = {
    'features_sample.csv': sample[[c for c in SAVE_COLUMNS if c in sample.columns]],
    'train.csv'          : train[[c  for c in SAVE_COLUMNS if c in train.columns]],
    'val.csv'            : val[[c    for c in SAVE_COLUMNS if c in val.columns]],
    'test.csv'           : test[[c   for c in SAVE_COLUMNS if c in test.columns]],
}
for fname, fdf in files.items():
    path = DATA_OUT / fname
    fdf.to_csv(path, index=False)
    print(f'  {path}  ({path.stat().st_size/1e6:.1f} MB)')

joblib.dump(le_platform,   MODEL_OUT / 'le_platform.pkl')
joblib.dump(le_sentiment,  MODEL_OUT / 'le_sentiment.pkl')
joblib.dump(scaler_standard, MODEL_OUT / 'scaler_standard.pkl')
joblib.dump(scaler_minmax,   MODEL_OUT / 'scaler_minmax.pkl')
joblib.dump(class_weights,   MODEL_OUT / 'class_weights.pkl')

maps = {
    'platform_to_id' : dict(zip(le_platform.classes_.tolist(),  range(len(le_platform.classes_)))),
    'id_to_platform' : dict(zip(range(len(le_platform.classes_)),  le_platform.classes_.tolist())),
    'sentiment_to_id': dict(zip(le_sentiment.classes_.tolist(), range(len(le_sentiment.classes_)))),
    'id_to_sentiment': dict(zip(range(len(le_sentiment.classes_)), le_sentiment.classes_.tolist()))
}
with open(MODEL_OUT / 'encoding_maps.json', 'w') as f:
    json.dump(maps, f, indent=2)
with open(MODEL_OUT / 'feature_registry.json', 'w') as f:
    json.dump({'numerical_features': NUMERICAL_FEATURES,
               'binary_features': BINARY_FEATURES,
               'all_model_features': ALL_MODEL_FEATURES,
               'n_features': len(ALL_MODEL_FEATURES),
               'n_platforms': int(df['platform_id'].nunique()),
               'created_at': datetime.now().isoformat()}, f, indent=2)

print('\n   le_platform.pkl')
print('   le_sentiment.pkl')
print('   scaler_standard.pkl')
print('   scaler_minmax.pkl')
print('   class_weights.pkl')
print('   encoding_maps.json')
print('   feature_registry.json')

  /kaggle/working/data/processed/features_sample.csv  (65.5 MB)
  /kaggle/working/data/processed/train.csv  (46.8 MB)
  /kaggle/working/data/processed/val.csv  (6.4 MB)
  /kaggle/working/data/processed/test.csv  (12.3 MB)

   le_platform.pkl
   le_sentiment.pkl
   scaler_standard.pkl
   scaler_minmax.pkl
   class_weights.pkl
   encoding_maps.json
   feature_registry.json


In [13]:
print(' /kaggle/working contents:\n')
for root, dirs, files in os.walk('/kaggle/working'):
    for file in sorted(files):
        path = os.path.join(root, file)
        mb   = os.path.getsize(path) / 1e6
        print(f'  {path}  ({mb:.1f} MB)')

 /kaggle/working contents:

  /kaggle/working/__notebook__.ipynb  (0.0 MB)
  /kaggle/working/data/processed/features_sample.csv  (65.5 MB)
  /kaggle/working/data/processed/test.csv  (12.3 MB)
  /kaggle/working/data/processed/train.csv  (46.8 MB)
  /kaggle/working/data/processed/val.csv  (6.4 MB)
  /kaggle/working/models/class_weights.pkl  (0.0 MB)
  /kaggle/working/models/encoding_maps.json  (0.0 MB)
  /kaggle/working/models/feature_registry.json  (0.0 MB)
  /kaggle/working/models/le_platform.pkl  (0.0 MB)
  /kaggle/working/models/le_sentiment.pkl  (0.0 MB)
  /kaggle/working/models/scaler_minmax.pkl  (0.0 MB)
  /kaggle/working/models/scaler_standard.pkl  (0.0 MB)
